In [9]:
# Install system dependencies
!apt-get update
!apt-get install -y ffmpeg

# Install Python packages
!pip install openai yt-dlp pydantic tenacity langgraph langchain-core langchain-openai ffmpeg-python


'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Cell 1: Setup (run once)
import sys
!pip install openai yt-dlp pydantic tenacity langgraph langchain-core python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

# Cell 2: Set API key
import os
os.environ['OPENAI_API_KEY'] = 'sk-proj....'

# Cell 3: Import and run
#!/usr/bin/env python3
"""
YouTube SEO Optimization Engine
Production-grade AI-powered SEO asset generator with multi-input support
"""

import os
import sys
import json
import logging
import tempfile
import time
import re
import subprocess
from pathlib import Path
from typing import Dict, Any, List, Optional, TypedDict, Tuple

import yt_dlp
from openai import OpenAI, OpenAIError, RateLimitError, APITimeoutError, APIConnectionError
from pydantic import BaseModel, Field, field_validator
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from langgraph.graph import StateGraph, END

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
else:
    from dotenv import load_dotenv
    load_dotenv()

# Configure logging to stderr
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stderr)],
)
logger = logging.getLogger(__name__)

# Constants
MAX_RETRIES = 3
MAX_TRANSCRIPT_CHARS = 48000
TARGET_COMPRESSED_CHARS = 32000
SUPPORTED_AUDIO_FORMATS = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg'}
SUPPORTED_VIDEO_FORMATS = {'.mp4', '.mov', '.mkv', '.avi', '.webm', '.flv'}


# ============================================================================
# PYDANTIC MODELS
# ============================================================================


class VideoMetadata(BaseModel):
    main_topic: str = Field(..., min_length=1)
    target_audience: str = Field(..., min_length=1)
    search_intent: str = Field(..., min_length=1)
    content_category: str = Field(..., min_length=1)
    duration_seconds: int = Field(..., gt=0)


class Keywords(BaseModel):
    primary: str = Field(..., min_length=1)
    secondary: List[str] = Field(..., min_length=5, max_length=10)
    long_tail: List[str] = Field(..., min_length=5, max_length=10)


class Titles(BaseModel):
    seo_titles: List[str] = Field(..., min_length=5, max_length=5)
    high_ctr_titles: List[str] = Field(..., min_length=3, max_length=3)

    @field_validator("seo_titles", "high_ctr_titles")
    @classmethod
    def validate_title_length(cls, v):
        validated = []
        for title in v:
            title = title.strip()
            if not title:
                raise ValueError("Title cannot be empty")
            if len(title) > 60:
                raise ValueError(f"Title exceeds 60 characters: {title}")
            validated.append(title)
        return validated


class Description(BaseModel):
    hook: str = Field(..., min_length=10)
    summary: str = Field(..., min_length=20)
    bullets: List[str] = Field(..., min_length=3, max_length=7)

    @field_validator("bullets")
    @classmethod
    def validate_bullets(cls, v):
        return [b.strip() for b in v if b.strip()]


class Chapter(BaseModel):
    timestamp: str = Field(..., pattern=r"^\d{1,2}:\d{2}(:\d{2})?$")
    title: str = Field(..., min_length=1)


class Assets(BaseModel):
    titles: Titles
    description: Description
    tags: List[str] = Field(..., min_length=15, max_length=25)
    chapters: List[Chapter] = Field(..., min_length=4, max_length=12)

    @field_validator("tags")
    @classmethod
    def validate_tags(cls, v):
        tags = [t.strip() for t in v if t.strip()]
        tags = list(dict.fromkeys(tags))
        if len(tags) < 15 or len(tags) > 25:
            raise ValueError(f"Tags must be 15-25 after normalization, got {len(tags)}")
        return tags

    @field_validator("chapters")
    @classmethod
    def validate_chapters(cls, v):
        if not v:
            raise ValueError("Chapters cannot be empty")

        first_ts = normalize_timestamp(v[0].timestamp)
        if first_ts != "00:00":
            raise ValueError(f"First chapter must start at 00:00, got {v[0].timestamp}")

        prev_seconds = 0
        for i, chapter in enumerate(v):
            normalized = normalize_timestamp(chapter.timestamp)
            seconds = timestamp_to_seconds(normalized)
            if i > 0 and seconds <= prev_seconds:
                raise ValueError(f"Timestamps must be strictly increasing at position {i}")
            prev_seconds = seconds
            v[i].timestamp = normalized

        return v


class SEOScore(BaseModel):
    overall: int = Field(..., ge=0, le=100)
    title_score: int = Field(..., ge=0, le=100)
    keyword_score: int = Field(..., ge=0, le=100)
    ctr_score: int = Field(..., ge=0, le=100)


class SEOOutput(BaseModel):
    video_metadata: VideoMetadata
    keywords: Keywords
    assets: Assets
    seo_score: SEOScore


# ============================================================================
# LANGGRAPH STATE
# ============================================================================


class AgentState(TypedDict):
    input_source: str
    file_path: Optional[str]
    youtube_url: Optional[str]
    duration_seconds: int
    transcript: str
    compressed_transcript: Optional[str]
    video_metadata: Optional[Dict[str, Any]]
    keywords: Optional[Dict[str, Any]]
    assets: Optional[Dict[str, Any]]
    seo_score: Optional[Dict[str, Any]]
    final_output: Optional[Dict[str, Any]]
    error: Optional[str]


# ============================================================================
# UTILITIES
# ============================================================================


def get_api_key() -> str:
    """Get OpenAI API key from environment."""
    if IN_COLAB:
        try:
            return userdata.get('OPENAI_API_KEY')
        except Exception:
            raise ValueError("OPENAI_API_KEY not found in Colab secrets")
    else:
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY environment variable is required")
        return api_key


def get_openai_client() -> OpenAI:
    """Initialize OpenAI client."""
    return OpenAI(api_key=get_api_key())


def get_chat_model() -> str:
    """Get chat model from environment."""
    return os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")


def get_stt_model() -> str:
    """Get STT model from environment."""
    return os.getenv("OPENAI_STT_MODEL", "whisper-1")


def normalize_timestamp(ts: str) -> str:
    """Normalize timestamp to mm:ss format."""
    ts = ts.strip()
    parts = ts.split(":")

    if len(parts) == 2:
        m, s = parts
        return f"{int(m):02d}:{int(s):02d}"
    elif len(parts) == 3:
        h, m, s = parts
        total_minutes = int(h) * 60 + int(m)
        return f"{total_minutes:02d}:{int(s):02d}"
    else:
        raise ValueError(f"Invalid timestamp format: {ts}")


def timestamp_to_seconds(ts: str) -> int:
    """Convert timestamp to seconds."""
    parts = ts.split(":")
    if len(parts) == 2:
        m, s = map(int, parts)
        return m * 60 + s
    elif len(parts) == 3:
        h, m, s = map(int, parts)
        return h * 3600 + m * 60 + s
    else:
        raise ValueError(f"Invalid timestamp: {ts}")


def seconds_to_timestamp(seconds: int) -> str:
    """Convert seconds to mm:ss format."""
    m = seconds // 60
    s = seconds % 60
    return f"{m:02d}:{s:02d}"


def extract_json_from_text(text: str) -> str:
    """Extract JSON from text."""
    text = text.strip()
    text = re.sub(r'^```json?\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE)
    text = text.strip()

    start = text.find('{')
    end = text.rfind('}')

    if start != -1 and end != -1 and end > start:
        return text[start:end+1]

    return text


def get_video_duration_ffmpeg(file_path: str) -> int:
    """Extract duration from video/audio file using ffmpeg."""
    try:
        cmd = [
            'ffprobe',
            '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            file_path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        duration = float(result.stdout.strip())
        return int(duration)
    except Exception as e:
        logger.warning(f"Failed to extract duration with ffprobe: {e}")
        return 600  # Default 10 minutes


def extract_audio_from_video(video_path: str, output_dir: Path) -> Path:
    """Extract audio from video file using ffmpeg."""
    logger.info(f"Extracting audio from video: {video_path}")
    output_path = output_dir / "extracted_audio.mp3"

    try:
        cmd = [
            'ffmpeg',
            '-i', video_path,
            '-vn',
            '-acodec', 'libmp3lame',
            '-q:a', '2',
            '-y',
            str(output_path)
        ]
        subprocess.run(cmd, capture_output=True, check=True)
        logger.info(f"Audio extracted: {output_path}")
        return output_path
    except subprocess.CalledProcessError as e:
        logger.error(f"ffmpeg extraction failed: {e.stderr.decode()}")
        raise RuntimeError(f"Failed to extract audio from video: {e}")


# ============================================================================
# INPUT HANDLING
# ============================================================================


def get_user_input() -> Tuple[str, Optional[str], Optional[str]]:
    """Get input from user: YouTube URL or file path."""
    print("\n" + "="*60)
    print("YouTube SEO Optimization Engine")
    print("="*60 + "\n")

    youtube_url = input("Enter YouTube URL (or press Enter to upload file): ").strip()

    if youtube_url:
        if not ('youtube.com' in youtube_url or 'youtu.be' in youtube_url):
            raise ValueError("Invalid YouTube URL. Must contain 'youtube.com' or 'youtu.be'")
        return "youtube", youtube_url, None

    print("\nNo YouTube URL provided. Please provide a file path.")
    file_path = input("Enter path to audio/video file: ").strip()

    if not file_path:
        raise ValueError("No input provided. Please provide either YouTube URL or file path.")

    file_path_obj = Path(file_path)
    if not file_path_obj.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    suffix = file_path_obj.suffix.lower()
    if suffix in SUPPORTED_AUDIO_FORMATS:
        return "audio_file", None, file_path
    elif suffix in SUPPORTED_VIDEO_FORMATS:
        return "video_file", None, file_path
    else:
        raise ValueError(f"Unsupported file format: {suffix}")


# ============================================================================
# YOUTUBE & FILE PROCESSING
# ============================================================================


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
def download_youtube_audio(youtube_url: str, output_dir: Path) -> Tuple[Path, int]:
    """Download audio from YouTube and return path + duration.

    Avoid ffmpeg post-processing so this works in environments without ffmpeg.
    """
    logger.info(f"Downloading audio from YouTube: {youtube_url}")
    start_time = time.time()

    ydl_opts = {
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": str(output_dir / "audio.%(ext)s"),
        "quiet": True,
        "no_warnings": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            duration = int(info.get('duration', 600))
            downloaded_path = Path(ydl.prepare_filename(info))
    except Exception as e:
        logger.error(f"yt-dlp download failed: {e}")
        raise

    if not downloaded_path.exists():
        candidates = sorted(output_dir.glob("audio.*"))
        if not candidates:
            raise FileNotFoundError("Downloaded audio file was not found")
        downloaded_path = candidates[0]

    elapsed = time.time() - start_time
    logger.info(
        f"Downloaded in {elapsed:.2f}s, duration: {duration}s, file: {downloaded_path.name}"
    )
    return downloaded_path, duration


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def transcribe_audio(client: OpenAI, audio_path: Path) -> str:
    """Transcribe audio using OpenAI Whisper."""
    logger.info(f"Transcribing audio: {audio_path}")
    start_time = time.time()

    stt_model = get_stt_model()

    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model=stt_model,
            file=audio_file,
            response_format="text"
        )

    elapsed = time.time() - start_time
    logger.info(f"Transcription completed in {elapsed:.2f}s: {len(transcript)} chars")
    return transcript


def preprocess_transcript(transcript: str) -> str:
    """Normalize whitespace."""
    lines = [line.strip() for line in transcript.split("\n") if line.strip()]
    cleaned = " ".join(lines)
    cleaned = " ".join(cleaned.split())
    return cleaned


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def compress_transcript(client: OpenAI, transcript: str, model: str, target_chars: int) -> str:
    """Compress transcript."""
    logger.info(f"Compressing transcript: {len(transcript)} -> ~{target_chars} chars")
    start_time = time.time()

    compression_ratio = target_chars / len(transcript)
    target_percentage = int(compression_ratio * 100)

    system_prompt = """You are an expert content analyst specializing in video transcript compression.

Your mission: Compress video transcripts into dense, information-rich summaries that preserve ALL critical information for SEO analysis.

Preserve:
- All key topics, concepts, and main ideas
- Important details, examples, and explanations
- Technical terms and specific information
- The overall structure and content flow
- Any timestamps or chapter markers

Create a comprehensive summary that captures 100% of the content's strategic value while reducing length significantly."""

    user_prompt = f"""Compress this video transcript into a detailed summary (approximately {target_percentage}% of original length) while preserving ALL important information:

{transcript}

Provide a dense, information-rich summary that maintains the content's full value for SEO analysis."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )

    compressed = response.choices[0].message.content.strip()
    elapsed = time.time() - start_time
    logger.info(f"Compressed in {elapsed:.2f}s: {len(compressed)} chars")
    return compressed


def prepare_transcript(transcript: str) -> str:
    """Prepare transcript, compressing if needed."""
    if len(transcript) <= MAX_TRANSCRIPT_CHARS:
        return transcript

    client = get_openai_client()
    model = get_chat_model()

    working = transcript
    for i in range(3):
        if len(working) <= MAX_TRANSCRIPT_CHARS:
            return working
        logger.info(f"Compression iteration {i+1}/3")
        working = compress_transcript(client, working, model, TARGET_COMPRESSED_CHARS)

    if len(working) > MAX_TRANSCRIPT_CHARS:
        logger.warning("Transcript still too long, truncating")
        working = working[:MAX_TRANSCRIPT_CHARS]

    return working


# ============================================================================
# LLM HELPERS
# ============================================================================


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def call_llm_json(client: OpenAI, model: str, system: str, user: str, temp: float) -> Dict[str, Any]:
    """Call LLM with JSON response."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temp,
        response_format={"type": "json_object"},
    )

    text = response.choices[0].message.content.strip()
    json_text = extract_json_from_text(text)

    try:
        return json.loads(json_text)
    except json.JSONDecodeError as e:
        logger.error(f"JSON parse error: {e}")
        raise


# ============================================================================
# LANGGRAPH NODES
# ============================================================================


def process_input_node(state: AgentState) -> AgentState:
    """Node: Process input (YouTube or file)."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()

        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_path = Path(tmpdir)

            if state["input_source"] == "youtube":
                audio_path, duration = download_youtube_audio(state["youtube_url"], tmp_path)

            elif state["input_source"] == "audio_file":
                audio_path = Path(state["file_path"])
                duration = get_video_duration_ffmpeg(str(audio_path))

            elif state["input_source"] == "video_file":
                duration = get_video_duration_ffmpeg(state["file_path"])
                audio_path = extract_audio_from_video(state["file_path"], tmp_path)
            else:
                raise ValueError(f"Unknown input source: {state['input_source']}")

            state["duration_seconds"] = duration
            logger.info(f"Video duration: {duration}s ({duration//60}m {duration%60}s)")

            transcript = transcribe_audio(client, audio_path)

        transcript = preprocess_transcript(transcript)
        state["transcript"] = transcript

        working = prepare_transcript(transcript)
        state["compressed_transcript"] = working

        elapsed = time.time() - start_time
        logger.info(f"Input processing completed in {elapsed:.2f}s")
        return state

    except Exception as e:
        logger.error(f"Input processing error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent1_content_brief(state: AgentState) -> AgentState:
    """Agent 1: Content Brief - Extract video metadata."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        duration = state["duration_seconds"]

        system_prompt = """You are an elite YouTube algorithm specialist and content strategist with deep expertise in video discoverability.

Think like the YouTube algorithm. Your analysis directly impacts ranking, reach, and revenue.

Mission: Extract strategic metadata that maximizes algorithmic favorability and user engagement.

Core Competencies:
- Identifying rankable main topics (not generic descriptions)
- Precision audience profiling (demographics, intent, expertise level)
- Search intent classification with ranking awareness
- YouTube taxonomy alignment for maximum discoverability

Output Requirements:
- main_topic: Specific, searchable, algorithm-friendly topic (avoid fluff words like "exploring", "discovering")
- target_audience: Detailed profile with pain points and search behavior
- search_intent: ONE of: informational, tutorial, review, comparison, problem-solving, story, entertainment
- content_category: YouTube's official categories
- duration_seconds: Video duration in seconds (provided)

Strict JSON output only. No commentary."""

        user_prompt = f"""Analyze this video for YouTube SEO optimization.

TRANSCRIPT:
{transcript}

VIDEO DURATION: {duration} seconds ({duration//60}m {duration%60}s)

Extract strategic metadata in EXACT JSON format:
{{
  "video_metadata": {{
    "main_topic": "specific rankable topic",
    "target_audience": "detailed audience profile with search behavior",
    "search_intent": "informational|tutorial|review|comparison|problem-solving|story|entertainment",
    "content_category": "YouTube category",
    "duration_seconds": {duration}
  }}
}}

Requirements:
- All fields non-empty
- main_topic must be specific and searchable
- duration_seconds must be {duration}

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.2)
                validated = VideoMetadata(**result["video_metadata"])
                state["video_metadata"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 1 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 1 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Provide metadata in EXACT JSON:
{{
  "video_metadata": {{
    "main_topic": "specific topic",
    "target_audience": "detailed profile",
    "search_intent": "informational|tutorial|review|comparison|problem-solving|story|entertainment",
    "content_category": "category",
    "duration_seconds": {duration}
  }}
}}

All fields required and non-empty. JSON only."""
                else:
                    raise

        raise ValueError("Agent 1 failed")

    except Exception as e:
        logger.error(f"Agent 1 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent2_keyword_strategy(state: AgentState) -> AgentState:
    """Agent 2: Keyword Strategy - Generate ranking-focused keywords."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        metadata = state["video_metadata"]

        system_prompt = """You are a world-class YouTube SEO keyword strategist obsessed with ranking and discoverability.

Think like YouTube's search algorithm. Focus on RANKING INTENT, not description.

Mission: Generate keyword strategies that drive rankings, traffic, and conversions.

Keyword Psychology:
- Primary: The money keyword - what users actively search to find this content
- Secondary: Related high-value search terms with ranking potential
- Long-tail: Specific user queries with lower competition, higher intent

Ranking Modifiers (USE THESE):
- "best", "top", "guide", "complete guide", "tutorial"
- "how to", "step by step", "tips", "tricks"
- "must visit", "hidden gems", "ultimate"
- Year/numbers for freshness

AVOID:
- Generic descriptors ("exploring", "discovering", "journey")
- Non-searchable phrases
- Overly academic language
- Marketing fluff

Strict JSON only."""

        user_prompt = f"""Generate a ranking-focused keyword strategy for this video.

VIDEO METADATA:
{json.dumps(metadata, indent=2)}

TRANSCRIPT:
{transcript}

Create keywords optimized for YouTube search rankings in EXACT JSON:
{{
  "keywords": {{
    "primary": "single high-value ranking keyword (2-5 words with modifiers)",
    "secondary": [
      "ranking keyword with modifier 1",
      "ranking keyword with modifier 2",
      "ranking keyword with modifier 3",
      "ranking keyword with modifier 4",
      "ranking keyword with modifier 5"
    ],
    "long_tail": [
      "how to [specific action] for [audience]",
      "best [topic] for [use case]",
      "top [number] [topic] [modifier]",
      "[topic] complete guide for [audience]",
      "[problem] solution using [topic]"
    ]
  }}
}}

Requirements:
- primary: 1 keyword with ranking modifiers
- secondary: 5-10 keywords (include "best", "top", "guide", numbers)
- long_tail: 5-10 question/search phrases (4-8 words)
- Focus on what users SEARCH, not what sounds nice

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.4)
                validated = Keywords(**result["keywords"])
                state["keywords"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 2 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 2 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Generate keywords in EXACT JSON:
{{
  "keywords": {{
    "primary": "keyword with modifier",
    "secondary": ["kw1", "kw2", "kw3", "kw4", "kw5"],
    "long_tail": ["phrase1", "phrase2", "phrase3", "phrase4", "phrase5"]
  }}
}}

- secondary: 5-10 items
- long_tail: 5-10 items

JSON only."""
                else:
                    raise

        raise ValueError("Agent 2 failed")

    except Exception as e:
        logger.error(f"Agent 2 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent3_seo_assets(state: AgentState) -> AgentState:
    """Agent 3: SEO Assets - Generate optimized titles, description, tags, chapters."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        metadata = state["video_metadata"]
        keywords = state["keywords"]
        duration = state["duration_seconds"]

        # Determine chapter count based on duration
        if duration < 180:  # < 3 minutes
            min_chapters = 4
            max_chapters = 6
        else:
            min_chapters = 6
            max_chapters = 10

        system_prompt = """You are an elite YouTube SEO copywriter obsessed with CTR, watch time, and algorithmic ranking.

Think like YouTube's recommendation algorithm + human psychology.

Mission: Create assets that MAXIMIZE clicks, engagement, and rankings.

TITLES - Critical Rules:
SEO Titles (5 required):
- Primary keyword in first 40 characters
- At least 2 titles must include numbers
- At least 3 titles must contain primary keyword
- Use power words: Best, Top, Ultimate, Complete, Must, Hidden
- 55-60 characters optimal
- AVOID: "Exploring", "Discovering", "Journey", "Unveiling"

High-CTR Titles (3 required):
- 1 curiosity-driven (without spoiling)
- 1 benefit-focused (clear value)
- 1 urgency/FOMO-driven
- 50-60 characters
- NOT clickbait spam

DESCRIPTION - Algorithm Rules:
- First 120 characters MUST contain primary keyword
- Include 1 natural search-style question
- Primary keyword appears 2-3 times naturally
- Include subtle CTA
- Professional, authoritative tone

TAGS:
- 15-25 unique tags
- Include primary + secondary keywords
- Include long-tail variations
- Mix broad + specific tags
- Deduplicate and normalize

CHAPTERS - Duration Awareness:
- MUST start at "00:00"
- CANNOT exceed video duration
- Evenly spaced throughout duration
- Format: mm:ss (e.g., "00:00", "02:45")
- Descriptive titles with keywords

Strict JSON only."""

        user_prompt = f"""Generate complete YouTube SEO assets optimized for ranking and CTR.

VIDEO METADATA:
{json.dumps(metadata, indent=2)}

KEYWORDS:
{json.dumps(keywords, indent=2)}

VIDEO DURATION: {duration} seconds ({duration//60}m {duration%60}s)

TRANSCRIPT:
{transcript}

Generate assets in EXACT JSON format:
{{
  "assets": {{
    "titles": {{
      "seo_titles": [
        "Title with primary keyword and number (55-60 chars)",
        "Title with primary keyword and power word",
        "Title with primary keyword and modifier",
        "Title with number and benefit",
        "Title with top/best and primary keyword"
      ],
      "high_ctr_titles": [
        "Curiosity-driven title (50-60 chars)",
        "Benefit-focused title with clear value",
        "FOMO/urgency title"
      ]
    }},
    "description": {{
      "hook": "First 120 chars with primary keyword: {keywords['primary']}. Include search question.",
      "summary": "Paragraph with primary keyword repeated 2-3 times naturally. Authoritative tone. Include CTA.",
      "bullets": [
        "Key point 1 with keyword",
        "Key point 2 with benefit",
        "Key point 3 with value",
        "Key point 4 with outcome"
      ]
    }},
    "tags": [
      "{keywords['primary']}",
      "related tag 1",
      "related tag 2",
      "15-25 total unique tags"
    ],
    "chapters": [
      {{"timestamp": "00:00", "title": "Introduction to [Topic]"}},
      {{"timestamp": "XX:XX", "title": "Chapter with keyword"}},
      "...{min_chapters}-{max_chapters} chapters total..."
    ]
  }}
}}

CRITICAL REQUIREMENTS:
- seo_titles: EXACTLY 5, max 60 chars each, at least 2 with numbers, at least 3 with primary keyword
- high_ctr_titles: EXACTLY 3, max 60 chars each
- description.hook: first 120 chars MUST contain "{keywords['primary']}"
- tags: 15-25 unique, deduplicated
- chapters: {min_chapters}-{max_chapters} chapters
- chapters[0].timestamp: MUST be "00:00"
- Last chapter timestamp: CANNOT exceed {seconds_to_timestamp(duration)} (duration limit)
- All timestamps: mm:ss format, strictly increasing

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.7)

                # Validate chapters don't exceed duration
                chapters = result["assets"]["chapters"]
                for chapter in chapters:
                    ts_seconds = timestamp_to_seconds(normalize_timestamp(chapter["timestamp"]))
                    if ts_seconds > duration:
                        raise ValueError(f"Chapter timestamp {chapter['timestamp']} exceeds video duration {duration}s")

                validated = Assets(**result["assets"])
                state["assets"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 3 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 3 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Regenerate assets in EXACT JSON format:
{{
  "assets": {{
    "titles": {{
      "seo_titles": [5 titles max 60 chars, 2+ with numbers, 3+ with primary keyword],
      "high_ctr_titles": [3 titles max 60 chars]
    }},
    "description": {{
      "hook": "first 120 chars with '{keywords['primary']}'",
      "summary": "paragraph",
      "bullets": [3-7 bullets]
    }},
    "tags": [15-25 unique tags],
    "chapters": [
      {{"timestamp": "00:00", "title": "..."}},
      ...{min_chapters}-{max_chapters} chapters total...
    ]
  }}
}}

CRITICAL:
- First chapter: "00:00"
- Last chapter: CANNOT exceed {seconds_to_timestamp(duration)}
- All timestamps: mm:ss, strictly increasing
- Duration limit: {duration} seconds

JSON only."""
                else:
                    raise

        raise ValueError("Agent 3 failed")

    except Exception as e:
        logger.error(f"Agent 3 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def calculate_seo_score(state: AgentState) -> AgentState:
    """Calculate SEO quality scores."""
    if state.get("error"):
        return state

    try:
        metadata = state["video_metadata"]
        keywords = state["keywords"]
        assets = state["assets"]

        # Title scoring
        title_score = 0
        primary_kw = keywords["primary"].lower()

        # Check SEO titles
        titles_with_numbers = sum(1 for t in assets["titles"]["seo_titles"] if re.search(r'\d', t))
        titles_with_primary = sum(1 for t in assets["titles"]["seo_titles"] if primary_kw in t.lower())

        title_score += min(titles_with_numbers * 10, 20)  # 20 points for 2+ numbers
        title_score += min(titles_with_primary * 10, 30)  # 30 points for 3+ primary keyword
        title_score += 25 if len(assets["titles"]["seo_titles"]) == 5 else 0
        title_score += 25 if len(assets["titles"]["high_ctr_titles"]) == 3 else 0

        # Keyword scoring
        keyword_score = 0
        keyword_score += 40 if keywords["primary"] else 0
        keyword_score += 30 if len(keywords["secondary"]) >= 5 else 0
        keyword_score += 30 if len(keywords["long_tail"]) >= 5 else 0

        # CTR scoring
        ctr_score = 0
        hook = assets["description"]["hook"].lower()
        ctr_score += 50 if primary_kw in hook[:120] else 0
        ctr_score += 25 if len(assets["tags"]) >= 15 else 0
        ctr_score += 25 if assets["chapters"][0]["timestamp"] == "00:00" else 0

        # Overall
        overall = int((title_score + keyword_score + ctr_score) / 3)

        seo_score = {
            "overall": overall,
            "title_score": title_score,
            "keyword_score": keyword_score,
            "ctr_score": ctr_score,
        }

        validated = SEOScore(**seo_score)
        state["seo_score"] = validated.model_dump()
        logger.info(f"SEO Score calculated: {overall}/100")
        return state

    except Exception as e:
        logger.error(f"SEO score calculation error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def compile_final_output(state: AgentState) -> AgentState:
    """Compile final JSON output."""
    if state.get("error"):
        return state

    try:
        final = {
            "video_metadata": state["video_metadata"],
            "keywords": state["keywords"],
            "assets": state["assets"],
            "seo_score": state["seo_score"],
        }

        validated = SEOOutput(**final)
        state["final_output"] = validated.model_dump()
        logger.info("Final output compiled")
        return state

    except Exception as e:
        logger.error(f"Final compilation error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


# ============================================================================
# WORKFLOW
# ============================================================================


def should_continue(state: AgentState) -> str:
    """Conditional routing."""
    return "error" if state.get("error") else "continue"


def create_workflow() -> StateGraph:
    """Create LangGraph workflow."""
    workflow = StateGraph(AgentState)

    workflow.add_node("process_input", process_input_node)
    workflow.add_node("agent1", agent1_content_brief)
    workflow.add_node("agent2", agent2_keyword_strategy)
    workflow.add_node("agent3", agent3_seo_assets)
    workflow.add_node("calculate_score", calculate_seo_score)
    workflow.add_node("compile", compile_final_output)

    workflow.set_entry_point("process_input")

    workflow.add_conditional_edges("process_input", should_continue, {"continue": "agent1", "error": END})
    workflow.add_conditional_edges("agent1", should_continue, {"continue": "agent2", "error": END})
    workflow.add_conditional_edges("agent2", should_continue, {"continue": "agent3", "error": END})
    workflow.add_conditional_edges("agent3", should_continue, {"continue": "calculate_score", "error": END})
    workflow.add_conditional_edges("calculate_score", should_continue, {"continue": "compile", "error": END})
    workflow.add_edge("compile", END)

    return workflow.compile()


# ============================================================================
# MAIN
# ============================================================================


def process_video(input_source: str, youtube_url: Optional[str], file_path: Optional[str]) -> Dict[str, Any]:
    """Main processing function."""
    logger.info("Starting YouTube SEO optimization")
    overall_start = time.time()

    initial_state: AgentState = {
        "input_source": input_source,
        "youtube_url": youtube_url,
        "file_path": file_path,
        "duration_seconds": 0,
        "transcript": "",
        "compressed_transcript": None,
        "video_metadata": None,
        "keywords": None,
        "assets": None,
        "seo_score": None,
        "final_output": None,
        "error": None,
    }

    app = create_workflow()
    final_state = app.invoke(initial_state)

    if final_state.get("error"):
        raise RuntimeError(f"Workflow failed: {final_state['error']}")

    if not final_state.get("final_output"):
        raise RuntimeError("No final output produced")

    elapsed = time.time() - overall_start
    logger.info(f"Total processing: {elapsed:.2f}s")

    return final_state["final_output"]


def main():
    """Entry point."""
    try:
        input_source, youtube_url, file_path = get_user_input()
        result = process_video(input_source, youtube_url, file_path)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        return 0

    except KeyboardInterrupt:
        logger.info("Interrupted by user")
        return 1
    except Exception as e:
        logger.error(f"Fatal error: {e}", exc_info=True)
        return 1


# if __name__ == "__main__":
#     main()

# Instead of the if __name__ == "__main__" block, use:
input_source, youtube_url, file_path = get_user_input()
result = process_video(input_source, youtube_url, file_path)
print(json.dumps(result, indent=2, ensure_ascii=False))


YouTube SEO Optimization Engine



2026-02-20 13:25:37,720 [INFO] Starting YouTube SEO optimization
2026-02-20 13:25:37,971 [INFO] Downloading audio from YouTube: https://youtu.be/nIYFa2YF55U?si=Wtx7a5dwJxXWjfK3


2026-02-20 13:25:44,338 [INFO] Downloaded in 6.37s, duration: 139s, file: audio.m4a
2026-02-20 13:25:44,338 [INFO] Video duration: 139s (2m 19s)
2026-02-20 13:25:44,338 [INFO] Transcribing audio: C:\Temp\tmp66qrcxis\audio.m4a
2026-02-20 13:26:19,491 [INFO] HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-02-20 13:26:19,491 [INFO] Transcription completed in 35.15s: 280 chars
2026-02-20 13:26:19,491 [INFO] Input processing completed in 41.75s
2026-02-20 13:26:22,674 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:26:22,691 [INFO] Agent 1 completed in 3.20s
2026-02-20 13:26:27,129 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:26:27,141 [INFO] Agent 2 completed in 4.43s
2026-02-20 13:26:38,375 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:26:38,391 [INFO] Agent 3 completed in 11.25s
2026-02

{
  "video_metadata": {
    "main_topic": "اسلامی ریپبلک پاکستان میں موسیقی کی تاریخ",
    "target_audience": "موسیقی کے شوقین، پاکستانی ثقافت میں دلچسپی رکھنے والے، طلباء اور محققین جو اسلامی موسیقی اور اس کی تاریخ کے بارے میں معلومات تلاش کر رہے ہیں",
    "search_intent": "informational",
    "content_category": "Education",
    "duration_seconds": 139
  },
  "keywords": {
    "primary": "best Islamic music history in Pakistan",
    "secondary": [
      "top Islamic music genres in Pakistan",
      "Islamic music complete guide",
      "best Pakistani music history",
      "top 10 Islamic music artists",
      "Islamic music tutorial for beginners",
      "must-know facts about Islamic music",
      "hidden gems of Islamic music in Pakistan",
      "Islamic music trends 2023"
    ],
    "long_tail": [
      "how to understand Islamic music for beginners",
      "best Islamic music for cultural appreciation",
      "top 5 Islamic music styles in Pakistan",
      "Islamic music history

In [15]:
#!/usr/bin/env python3
"""
YouTube SEO Optimization Engine
Production-grade AI-powered SEO asset generator with multi-input support
"""

import os
import sys
import json
import logging
import tempfile
import time
import re
import subprocess
from pathlib import Path
from typing import Dict, Any, List, Optional, TypedDict, Tuple

import yt_dlp
from openai import OpenAI, OpenAIError, RateLimitError, APITimeoutError, APIConnectionError
from pydantic import BaseModel, Field, field_validator
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from langgraph.graph import StateGraph, END

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
else:
    from dotenv import load_dotenv
    load_dotenv()

# Configure logging to stderr
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stderr)],
)
logger = logging.getLogger(__name__)

# Constants
MAX_RETRIES = 3
MAX_TRANSCRIPT_CHARS = 48000
TARGET_COMPRESSED_CHARS = 32000
SUPPORTED_AUDIO_FORMATS = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg'}
SUPPORTED_VIDEO_FORMATS = {'.mp4', '.mov', '.mkv', '.avi', '.webm', '.flv'}


# ============================================================================
# PYDANTIC MODELS
# ============================================================================


class VideoMetadata(BaseModel):
    main_topic: str = Field(..., min_length=1)
    target_audience: str = Field(..., min_length=1)
    search_intent: str = Field(..., min_length=1)
    content_category: str = Field(..., min_length=1)
    duration_seconds: int = Field(..., gt=0)


class Keywords(BaseModel):
    primary: str = Field(..., min_length=1)
    secondary: List[str] = Field(..., min_length=5, max_length=10)
    long_tail: List[str] = Field(..., min_length=5, max_length=10)


class Titles(BaseModel):
    seo_titles: List[str] = Field(..., min_length=5, max_length=5)
    high_ctr_titles: List[str] = Field(..., min_length=3, max_length=3)

    @field_validator("seo_titles", "high_ctr_titles")
    @classmethod
    def validate_title_length(cls, v):
        validated = []
        for title in v:
            title = title.strip()
            if not title:
                raise ValueError("Title cannot be empty")
            if len(title) > 60:
                raise ValueError(f"Title exceeds 60 characters: {title}")
            validated.append(title)
        return validated


class Description(BaseModel):
    hook: str = Field(..., min_length=10)
    summary: str = Field(..., min_length=20)
    bullets: List[str] = Field(..., min_length=3, max_length=7)

    @field_validator("bullets")
    @classmethod
    def validate_bullets(cls, v):
        return [b.strip() for b in v if b.strip()]


class Chapter(BaseModel):
    timestamp: str = Field(..., pattern=r"^\d{1,2}:\d{2}(:\d{2})?$")
    title: str = Field(..., min_length=1)


class Assets(BaseModel):
    titles: Titles
    description: Description
    tags: List[str] = Field(..., min_length=15, max_length=25)
    chapters: List[Chapter] = Field(..., min_length=4, max_length=12)

    @field_validator("tags")
    @classmethod
    def validate_tags(cls, v):
        tags = [t.strip() for t in v if t.strip()]
        tags = list(dict.fromkeys(tags))
        if len(tags) < 15 or len(tags) > 25:
            raise ValueError(f"Tags must be 15-25 after normalization, got {len(tags)}")
        return tags

    @field_validator("chapters")
    @classmethod
    def validate_chapters(cls, v):
        if not v:
            raise ValueError("Chapters cannot be empty")

        first_ts = normalize_timestamp(v[0].timestamp)
        if first_ts != "00:00":
            raise ValueError(f"First chapter must start at 00:00, got {v[0].timestamp}")

        prev_seconds = 0
        for i, chapter in enumerate(v):
            normalized = normalize_timestamp(chapter.timestamp)
            seconds = timestamp_to_seconds(normalized)
            if i > 0 and seconds <= prev_seconds:
                raise ValueError(f"Timestamps must be strictly increasing at position {i}")
            prev_seconds = seconds
            v[i].timestamp = normalized

        return v


class SEOScore(BaseModel):
    overall: int = Field(..., ge=0, le=100)
    title_score: int = Field(..., ge=0, le=100)
    keyword_score: int = Field(..., ge=0, le=100)
    ctr_score: int = Field(..., ge=0, le=100)


class SEOOutput(BaseModel):
    video_metadata: VideoMetadata
    keywords: Keywords
    assets: Assets
    seo_score: SEOScore


# ============================================================================
# LANGGRAPH STATE
# ============================================================================


class AgentState(TypedDict):
    input_source: str
    file_path: Optional[str]
    youtube_url: Optional[str]
    duration_seconds: int
    transcript: str
    compressed_transcript: Optional[str]
    video_metadata: Optional[Dict[str, Any]]
    keywords: Optional[Dict[str, Any]]
    assets: Optional[Dict[str, Any]]
    seo_score: Optional[Dict[str, Any]]
    final_output: Optional[Dict[str, Any]]
    error: Optional[str]


# ============================================================================
# UTILITIES
# ============================================================================


def get_api_key() -> str:
    """Get OpenAI API key from environment."""
    if IN_COLAB:
        try:
            return userdata.get('OPENAI_API_KEY')
        except Exception:
            raise ValueError("OPENAI_API_KEY not found in Colab secrets")
    else:
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY environment variable is required")
        return api_key


def get_openai_client() -> OpenAI:
    """Initialize OpenAI client."""
    return OpenAI(api_key=get_api_key())


def get_chat_model() -> str:
    """Get chat model from environment."""
    return os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")


def get_stt_model() -> str:
    """Get STT model from environment."""
    return os.getenv("OPENAI_STT_MODEL", "whisper-1")


def normalize_timestamp(ts: str) -> str:
    """Normalize timestamp to mm:ss format."""
    ts = ts.strip()
    parts = ts.split(":")

    if len(parts) == 2:
        m, s = parts
        return f"{int(m):02d}:{int(s):02d}"
    elif len(parts) == 3:
        h, m, s = parts
        total_minutes = int(h) * 60 + int(m)
        return f"{total_minutes:02d}:{int(s):02d}"
    else:
        raise ValueError(f"Invalid timestamp format: {ts}")


def timestamp_to_seconds(ts: str) -> int:
    """Convert timestamp to seconds."""
    parts = ts.split(":")
    if len(parts) == 2:
        m, s = map(int, parts)
        return m * 60 + s
    elif len(parts) == 3:
        h, m, s = map(int, parts)
        return h * 3600 + m * 60 + s
    else:
        raise ValueError(f"Invalid timestamp: {ts}")


def seconds_to_timestamp(seconds: int) -> str:
    """Convert seconds to mm:ss format."""
    m = seconds // 60
    s = seconds % 60
    return f"{m:02d}:{s:02d}"


def extract_json_from_text(text: str) -> str:
    """Extract JSON from text."""
    text = text.strip()
    text = re.sub(r'^```json?\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*```$', '', text, flags=re.MULTILINE)
    text = text.strip()

    start = text.find('{')
    end = text.rfind('}')

    if start != -1 and end != -1 and end > start:
        return text[start:end+1]

    return text


def get_video_duration_ffmpeg(file_path: str) -> int:
    """Extract duration from video/audio file using ffmpeg."""
    try:
        cmd = [
            'ffprobe',
            '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            file_path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        duration = float(result.stdout.strip())
        return int(duration)
    except Exception as e:
        logger.warning(f"Failed to extract duration with ffprobe: {e}")
        return 600  # Default 10 minutes


def extract_audio_from_video(video_path: str, output_dir: Path) -> Path:
    """Extract audio from video file using ffmpeg."""
    logger.info(f"Extracting audio from video: {video_path}")
    output_path = output_dir / "extracted_audio.mp3"

    try:
        cmd = [
            'ffmpeg',
            '-i', video_path,
            '-vn',
            '-acodec', 'libmp3lame',
            '-q:a', '2',
            '-y',
            str(output_path)
        ]
        subprocess.run(cmd, capture_output=True, check=True)
        logger.info(f"Audio extracted: {output_path}")
        return output_path
    except subprocess.CalledProcessError as e:
        logger.error(f"ffmpeg extraction failed: {e.stderr.decode()}")
        raise RuntimeError(f"Failed to extract audio from video: {e}")


# ============================================================================
# INPUT HANDLING
# ============================================================================


def get_user_input() -> Tuple[str, Optional[str], Optional[str]]:
    """Get input from user: YouTube URL or file path."""
    print("\n" + "="*60)
    print("YouTube SEO Optimization Engine")
    print("="*60 + "\n")

    youtube_url = input("Enter YouTube URL (or press Enter to upload file): ").strip()

    if youtube_url:
        if not ('youtube.com' in youtube_url or 'youtu.be' in youtube_url):
            raise ValueError("Invalid YouTube URL. Must contain 'youtube.com' or 'youtu.be'")
        return "youtube", youtube_url, None

    print("\nNo YouTube URL provided. Please provide a file path.")
    file_path = input("Enter path to audio/video file: ").strip()

    if not file_path:
        raise ValueError("No input provided. Please provide either YouTube URL or file path.")

    file_path_obj = Path(file_path)
    if not file_path_obj.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    suffix = file_path_obj.suffix.lower()
    if suffix in SUPPORTED_AUDIO_FORMATS:
        return "audio_file", None, file_path
    elif suffix in SUPPORTED_VIDEO_FORMATS:
        return "video_file", None, file_path
    else:
        raise ValueError(f"Unsupported file format: {suffix}")


# ============================================================================
# YOUTUBE & FILE PROCESSING
# ============================================================================


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
def download_youtube_audio(youtube_url: str, output_dir: Path) -> Tuple[Path, int]:
    """Download audio from YouTube and return path + duration.

    Avoid ffmpeg post-processing so this works in environments without ffmpeg.
    """
    logger.info(f"Downloading audio from YouTube: {youtube_url}")
    start_time = time.time()

    ydl_opts = {
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": str(output_dir / "audio.%(ext)s"),
        "quiet": True,
        "no_warnings": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            duration = int(info.get('duration', 600))
            downloaded_path = Path(ydl.prepare_filename(info))
    except Exception as e:
        logger.error(f"yt-dlp download failed: {e}")
        raise

    if not downloaded_path.exists():
        candidates = sorted(output_dir.glob("audio.*"))
        if not candidates:
            raise FileNotFoundError("Downloaded audio file was not found")
        downloaded_path = candidates[0]

    elapsed = time.time() - start_time
    logger.info(
        f"Downloaded in {elapsed:.2f}s, duration: {duration}s, file: {downloaded_path.name}"
    )
    return downloaded_path, duration


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def transcribe_audio(client: OpenAI, audio_path: Path) -> str:
    """Transcribe audio using OpenAI Whisper."""
    logger.info(f"Transcribing audio: {audio_path}")
    start_time = time.time()

    stt_model = get_stt_model()

    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model=stt_model,
            file=audio_file,
            response_format="text"
        )

    elapsed = time.time() - start_time
    logger.info(f"Transcription completed in {elapsed:.2f}s: {len(transcript)} chars")
    return transcript


def preprocess_transcript(transcript: str) -> str:
    """Normalize whitespace."""
    lines = [line.strip() for line in transcript.split("\n") if line.strip()]
    cleaned = " ".join(lines)
    cleaned = " ".join(cleaned.split())
    return cleaned


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def compress_transcript(client: OpenAI, transcript: str, model: str, target_chars: int) -> str:
    """Compress transcript."""
    logger.info(f"Compressing transcript: {len(transcript)} -> ~{target_chars} chars")
    start_time = time.time()

    compression_ratio = target_chars / len(transcript)
    target_percentage = int(compression_ratio * 100)

    system_prompt = """You are an expert content analyst specializing in video transcript compression.

Your mission: Compress video transcripts into dense, information-rich summaries that preserve ALL critical information for SEO analysis.

Preserve:
- All key topics, concepts, and main ideas
- Important details, examples, and explanations
- Technical terms and specific information
- The overall structure and content flow
- Any timestamps or chapter markers

Create a comprehensive summary that captures 100% of the content's strategic value while reducing length significantly."""

    user_prompt = f"""Compress this video transcript into a detailed summary (approximately {target_percentage}% of original length) while preserving ALL important information:

{transcript}

Provide a dense, information-rich summary that maintains the content's full value for SEO analysis."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )

    compressed = response.choices[0].message.content.strip()
    elapsed = time.time() - start_time
    logger.info(f"Compressed in {elapsed:.2f}s: {len(compressed)} chars")
    return compressed


def prepare_transcript(transcript: str) -> str:
    """Prepare transcript, compressing if needed."""
    if len(transcript) <= MAX_TRANSCRIPT_CHARS:
        return transcript

    client = get_openai_client()
    model = get_chat_model()

    working = transcript
    for i in range(3):
        if len(working) <= MAX_TRANSCRIPT_CHARS:
            return working
        logger.info(f"Compression iteration {i+1}/3")
        working = compress_transcript(client, working, model, TARGET_COMPRESSED_CHARS)

    if len(working) > MAX_TRANSCRIPT_CHARS:
        logger.warning("Transcript still too long, truncating")
        working = working[:MAX_TRANSCRIPT_CHARS]

    return working


# ============================================================================
# LLM HELPERS
# ============================================================================


@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError, OpenAIError)),
)
def call_llm_json(client: OpenAI, model: str, system: str, user: str, temp: float) -> Dict[str, Any]:
    """Call LLM with JSON response."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temp,
        response_format={"type": "json_object"},
    )

    text = response.choices[0].message.content.strip()
    json_text = extract_json_from_text(text)

    try:
        return json.loads(json_text)
    except json.JSONDecodeError as e:
        logger.error(f"JSON parse error: {e}")
        raise


# ============================================================================
# LANGGRAPH NODES
# ============================================================================


def process_input_node(state: AgentState) -> AgentState:
    """Node: Process input (YouTube or file)."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()

        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_path = Path(tmpdir)

            if state["input_source"] == "youtube":
                audio_path, duration = download_youtube_audio(state["youtube_url"], tmp_path)

            elif state["input_source"] == "audio_file":
                audio_path = Path(state["file_path"])
                duration = get_video_duration_ffmpeg(str(audio_path))

            elif state["input_source"] == "video_file":
                duration = get_video_duration_ffmpeg(state["file_path"])
                audio_path = extract_audio_from_video(state["file_path"], tmp_path)
            else:
                raise ValueError(f"Unknown input source: {state['input_source']}")

            state["duration_seconds"] = duration
            logger.info(f"Video duration: {duration}s ({duration//60}m {duration%60}s)")

            transcript = transcribe_audio(client, audio_path)

        transcript = preprocess_transcript(transcript)
        state["transcript"] = transcript

        working = prepare_transcript(transcript)
        state["compressed_transcript"] = working

        elapsed = time.time() - start_time
        logger.info(f"Input processing completed in {elapsed:.2f}s")
        return state

    except Exception as e:
        logger.error(f"Input processing error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent1_content_brief(state: AgentState) -> AgentState:
    """Agent 1: Content Brief - Extract video metadata."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        duration = state["duration_seconds"]

        system_prompt = """You are an elite YouTube algorithm specialist and content strategist with deep expertise in video discoverability.

Think like the YouTube algorithm. Your analysis directly impacts ranking, reach, and revenue.

Mission: Extract strategic metadata that maximizes algorithmic favorability and user engagement.

Core Competencies:
- Identifying rankable main topics (not generic descriptions)
- Precision audience profiling (demographics, intent, expertise level)
- Search intent classification with ranking awareness
- YouTube taxonomy alignment for maximum discoverability

Output Requirements:
- main_topic: Specific, searchable, algorithm-friendly topic (avoid fluff words like "exploring", "discovering")
- target_audience: Detailed profile with pain points and search behavior
- search_intent: ONE of: informational, tutorial, review, comparison, problem-solving, story, entertainment
- content_category: YouTube's official categories
- duration_seconds: Video duration in seconds (provided)

Strict JSON output only. No commentary."""

        user_prompt = f"""Analyze this video for YouTube SEO optimization.

TRANSCRIPT:
{transcript}

VIDEO DURATION: {duration} seconds ({duration//60}m {duration%60}s)

Extract strategic metadata in EXACT JSON format:
{{
  "video_metadata": {{
    "main_topic": "specific rankable topic",
    "target_audience": "detailed audience profile with search behavior",
    "search_intent": "informational|tutorial|review|comparison|problem-solving|story|entertainment",
    "content_category": "YouTube category",
    "duration_seconds": {duration}
  }}
}}

Requirements:
- All fields non-empty
- main_topic must be specific and searchable
- duration_seconds must be {duration}

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.2)
                validated = VideoMetadata(**result["video_metadata"])
                state["video_metadata"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 1 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 1 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Provide metadata in EXACT JSON:
{{
  "video_metadata": {{
    "main_topic": "specific topic",
    "target_audience": "detailed profile",
    "search_intent": "informational|tutorial|review|comparison|problem-solving|story|entertainment",
    "content_category": "category",
    "duration_seconds": {duration}
  }}
}}

All fields required and non-empty. JSON only."""
                else:
                    raise

        raise ValueError("Agent 1 failed")

    except Exception as e:
        logger.error(f"Agent 1 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent2_keyword_strategy(state: AgentState) -> AgentState:
    """Agent 2: Keyword Strategy - Generate ranking-focused keywords."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        metadata = state["video_metadata"]

        system_prompt = """You are a world-class YouTube SEO keyword strategist obsessed with ranking and discoverability.

Think like YouTube's search algorithm. Focus on RANKING INTENT, not description.

Mission: Generate keyword strategies that drive rankings, traffic, and conversions.

Keyword Psychology:
- Primary: The money keyword - what users actively search to find this content
- Secondary: Related high-value search terms with ranking potential
- Long-tail: Specific user queries with lower competition, higher intent

Ranking Modifiers (USE THESE):
- "best", "top", "guide", "complete guide", "tutorial"
- "how to", "step by step", "tips", "tricks"
- "must visit", "hidden gems", "ultimate"
- Year/numbers for freshness

AVOID:
- Generic descriptors ("exploring", "discovering", "journey")
- Non-searchable phrases
- Overly academic language
- Marketing fluff

Strict JSON only."""

        user_prompt = f"""Generate a ranking-focused keyword strategy for this video.

VIDEO METADATA:
{json.dumps(metadata, indent=2)}

TRANSCRIPT:
{transcript}

Create keywords optimized for YouTube search rankings in EXACT JSON:
{{
  "keywords": {{
    "primary": "single high-value ranking keyword (2-5 words with modifiers)",
    "secondary": [
      "ranking keyword with modifier 1",
      "ranking keyword with modifier 2",
      "ranking keyword with modifier 3",
      "ranking keyword with modifier 4",
      "ranking keyword with modifier 5"
    ],
    "long_tail": [
      "how to [specific action] for [audience]",
      "best [topic] for [use case]",
      "top [number] [topic] [modifier]",
      "[topic] complete guide for [audience]",
      "[problem] solution using [topic]"
    ]
  }}
}}

Requirements:
- primary: 1 keyword with ranking modifiers
- secondary: 5-10 keywords (include "best", "top", "guide", numbers)
- long_tail: 5-10 question/search phrases (4-8 words)
- Focus on what users SEARCH, not what sounds nice

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.4)
                validated = Keywords(**result["keywords"])
                state["keywords"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 2 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 2 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Generate keywords in EXACT JSON:
{{
  "keywords": {{
    "primary": "keyword with modifier",
    "secondary": ["kw1", "kw2", "kw3", "kw4", "kw5"],
    "long_tail": ["phrase1", "phrase2", "phrase3", "phrase4", "phrase5"]
  }}
}}

- secondary: 5-10 items
- long_tail: 5-10 items

JSON only."""
                else:
                    raise

        raise ValueError("Agent 2 failed")

    except Exception as e:
        logger.error(f"Agent 2 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def agent3_seo_assets(state: AgentState) -> AgentState:
    """Agent 3: SEO Assets - Generate optimized titles, description, tags, chapters."""
    if state.get("error"):
        return state

    try:
        start_time = time.time()
        client = get_openai_client()
        model = get_chat_model()
        transcript = state["compressed_transcript"]
        metadata = state["video_metadata"]
        keywords = state["keywords"]
        duration = state["duration_seconds"]

        # Determine chapter count based on duration
        if duration < 180:  # < 3 minutes
            min_chapters = 4
            max_chapters = 6
        else:
            min_chapters = 6
            max_chapters = 10

        system_prompt = """You are an elite YouTube SEO copywriter obsessed with CTR, watch time, and algorithmic ranking.

Think like YouTube's recommendation algorithm + human psychology.

Mission: Create assets that MAXIMIZE clicks, engagement, and rankings.

TITLES - Critical Rules:
SEO Titles (5 required):
- Primary keyword in first 40 characters
- At least 2 titles must include numbers
- At least 3 titles must contain primary keyword
- Use power words: Best, Top, Ultimate, Complete, Must, Hidden
- 55-60 characters optimal
- AVOID: "Exploring", "Discovering", "Journey", "Unveiling"

High-CTR Titles (3 required):
- 1 curiosity-driven (without spoiling)
- 1 benefit-focused (clear value)
- 1 urgency/FOMO-driven
- 50-60 characters
- NOT clickbait spam

DESCRIPTION - Algorithm Rules:
- First 120 characters MUST contain primary keyword
- Include 1 natural search-style question
- Primary keyword appears 2-3 times naturally
- Include subtle CTA
- Professional, authoritative tone

TAGS:
- 15-25 unique tags
- Include primary + secondary keywords
- Include long-tail variations
- Mix broad + specific tags
- Deduplicate and normalize

CHAPTERS - Duration Awareness:
- MUST start at "00:00"
- CANNOT exceed video duration
- Evenly spaced throughout duration
- Format: mm:ss (e.g., "00:00", "02:45")
- Descriptive titles with keywords

Strict JSON only."""

        user_prompt = f"""Generate complete YouTube SEO assets optimized for ranking and CTR.

VIDEO METADATA:
{json.dumps(metadata, indent=2)}

KEYWORDS:
{json.dumps(keywords, indent=2)}

VIDEO DURATION: {duration} seconds ({duration//60}m {duration%60}s)

TRANSCRIPT:
{transcript}

Generate assets in EXACT JSON format:
{{
  "assets": {{
    "titles": {{
      "seo_titles": [
        "Title with primary keyword and number (55-60 chars)",
        "Title with primary keyword and power word",
        "Title with primary keyword and modifier",
        "Title with number and benefit",
        "Title with top/best and primary keyword"
      ],
      "high_ctr_titles": [
        "Curiosity-driven title (50-60 chars)",
        "Benefit-focused title with clear value",
        "FOMO/urgency title"
      ]
    }},
    "description": {{
      "hook": "First 120 chars with primary keyword: {keywords['primary']}. Include search question.",
      "summary": "Paragraph with primary keyword repeated 2-3 times naturally. Authoritative tone. Include CTA.",
      "bullets": [
        "Key point 1 with keyword",
        "Key point 2 with benefit",
        "Key point 3 with value",
        "Key point 4 with outcome"
      ]
    }},
    "tags": [
      "{keywords['primary']}",
      "related tag 1",
      "related tag 2",
      "15-25 total unique tags"
    ],
    "chapters": [
      {{"timestamp": "00:00", "title": "Introduction to [Topic]"}},
      {{"timestamp": "XX:XX", "title": "Chapter with keyword"}},
      "...{min_chapters}-{max_chapters} chapters total..."
    ]
  }}
}}

CRITICAL REQUIREMENTS:
- seo_titles: EXACTLY 5, max 60 chars each, at least 2 with numbers, at least 3 with primary keyword
- high_ctr_titles: EXACTLY 3, max 60 chars each
- description.hook: first 120 chars MUST contain "{keywords['primary']}"
- tags: 15-25 unique, deduplicated
- chapters: {min_chapters}-{max_chapters} chapters
- chapters[0].timestamp: MUST be "00:00"
- Last chapter timestamp: CANNOT exceed {seconds_to_timestamp(duration)} (duration limit)
- All timestamps: mm:ss format, strictly increasing

JSON only."""

        for attempt in range(MAX_RETRIES):
            try:
                result = call_llm_json(client, model, system_prompt, user_prompt, 0.7)

                # Validate chapters don't exceed duration
                chapters = result["assets"]["chapters"]
                for chapter in chapters:
                    ts_seconds = timestamp_to_seconds(normalize_timestamp(chapter["timestamp"]))
                    if ts_seconds > duration:
                        raise ValueError(f"Chapter timestamp {chapter['timestamp']} exceeds video duration {duration}s")

                validated = Assets(**result["assets"])
                state["assets"] = validated.model_dump()

                elapsed = time.time() - start_time
                logger.info(f"Agent 3 completed in {elapsed:.2f}s")
                return state

            except (KeyError, ValueError) as e:
                logger.warning(f"Agent 3 validation failed (attempt {attempt+1}): {e}")
                if attempt < MAX_RETRIES - 1:
                    user_prompt = f"""Validation error: {e}

Regenerate assets in EXACT JSON format:
{{
  "assets": {{
    "titles": {{
      "seo_titles": [5 titles max 60 chars, 2+ with numbers, 3+ with primary keyword],
      "high_ctr_titles": [3 titles max 60 chars]
    }},
    "description": {{
      "hook": "first 120 chars with '{keywords['primary']}'",
      "summary": "paragraph",
      "bullets": [3-7 bullets]
    }},
    "tags": [15-25 unique tags],
    "chapters": [
      {{"timestamp": "00:00", "title": "..."}},
      ...{min_chapters}-{max_chapters} chapters total...
    ]
  }}
}}

CRITICAL:
- First chapter: "00:00"
- Last chapter: CANNOT exceed {seconds_to_timestamp(duration)}
- All timestamps: mm:ss, strictly increasing
- Duration limit: {duration} seconds

JSON only."""
                else:
                    raise

        raise ValueError("Agent 3 failed")

    except Exception as e:
        logger.error(f"Agent 3 error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def calculate_seo_score(state: AgentState) -> AgentState:
    """Calculate SEO quality scores."""
    if state.get("error"):
        return state

    try:
        metadata = state["video_metadata"]
        keywords = state["keywords"]
        assets = state["assets"]

        # Title scoring
        title_score = 0
        primary_kw = keywords["primary"].lower()

        # Check SEO titles
        titles_with_numbers = sum(1 for t in assets["titles"]["seo_titles"] if re.search(r'\d', t))
        titles_with_primary = sum(1 for t in assets["titles"]["seo_titles"] if primary_kw in t.lower())

        title_score += min(titles_with_numbers * 10, 20)  # 20 points for 2+ numbers
        title_score += min(titles_with_primary * 10, 30)  # 30 points for 3+ primary keyword
        title_score += 25 if len(assets["titles"]["seo_titles"]) == 5 else 0
        title_score += 25 if len(assets["titles"]["high_ctr_titles"]) == 3 else 0

        # Keyword scoring
        keyword_score = 0
        keyword_score += 40 if keywords["primary"] else 0
        keyword_score += 30 if len(keywords["secondary"]) >= 5 else 0
        keyword_score += 30 if len(keywords["long_tail"]) >= 5 else 0

        # CTR scoring
        ctr_score = 0
        hook = assets["description"]["hook"].lower()
        ctr_score += 50 if primary_kw in hook[:120] else 0
        ctr_score += 25 if len(assets["tags"]) >= 15 else 0
        ctr_score += 25 if assets["chapters"][0]["timestamp"] == "00:00" else 0

        # Overall
        overall = int((title_score + keyword_score + ctr_score) / 3)

        seo_score = {
            "overall": overall,
            "title_score": title_score,
            "keyword_score": keyword_score,
            "ctr_score": ctr_score,
        }

        validated = SEOScore(**seo_score)
        state["seo_score"] = validated.model_dump()
        logger.info(f"SEO Score calculated: {overall}/100")
        return state

    except Exception as e:
        logger.error(f"SEO score calculation error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


def compile_final_output(state: AgentState) -> AgentState:
    """Compile final JSON output."""
    if state.get("error"):
        return state

    try:
        final = {
            "video_metadata": state["video_metadata"],
            "keywords": state["keywords"],
            "assets": state["assets"],
            "seo_score": state["seo_score"],
        }

        validated = SEOOutput(**final)
        state["final_output"] = validated.model_dump()
        logger.info("Final output compiled")
        return state

    except Exception as e:
        logger.error(f"Final compilation error: {e}", exc_info=True)
        state["error"] = str(e)
        return state


# ============================================================================
# WORKFLOW
# ============================================================================


def should_continue(state: AgentState) -> str:
    """Conditional routing."""
    return "error" if state.get("error") else "continue"


def create_workflow() -> StateGraph:
    """Create LangGraph workflow."""
    workflow = StateGraph(AgentState)

    workflow.add_node("process_input", process_input_node)
    workflow.add_node("agent1", agent1_content_brief)
    workflow.add_node("agent2", agent2_keyword_strategy)
    workflow.add_node("agent3", agent3_seo_assets)
    workflow.add_node("calculate_score", calculate_seo_score)
    workflow.add_node("compile", compile_final_output)

    workflow.set_entry_point("process_input")

    workflow.add_conditional_edges("process_input", should_continue, {"continue": "agent1", "error": END})
    workflow.add_conditional_edges("agent1", should_continue, {"continue": "agent2", "error": END})
    workflow.add_conditional_edges("agent2", should_continue, {"continue": "agent3", "error": END})
    workflow.add_conditional_edges("agent3", should_continue, {"continue": "calculate_score", "error": END})
    workflow.add_conditional_edges("calculate_score", should_continue, {"continue": "compile", "error": END})
    workflow.add_edge("compile", END)

    return workflow.compile()


# ============================================================================
# MAIN
# ============================================================================


def process_video(input_source: str, youtube_url: Optional[str], file_path: Optional[str]) -> Dict[str, Any]:
    """Main processing function."""
    logger.info("Starting YouTube SEO optimization")
    overall_start = time.time()

    initial_state: AgentState = {
        "input_source": input_source,
        "youtube_url": youtube_url,
        "file_path": file_path,
        "duration_seconds": 0,
        "transcript": "",
        "compressed_transcript": None,
        "video_metadata": None,
        "keywords": None,
        "assets": None,
        "seo_score": None,
        "final_output": None,
        "error": None,
    }

    app = create_workflow()
    final_state = app.invoke(initial_state)

    if final_state.get("error"):
        raise RuntimeError(f"Workflow failed: {final_state['error']}")

    if not final_state.get("final_output"):
        raise RuntimeError("No final output produced")

    elapsed = time.time() - overall_start
    logger.info(f"Total processing: {elapsed:.2f}s")

    return final_state["final_output"]


def main():
    """Entry point."""
    try:
        input_source, youtube_url, file_path = get_user_input()
        result = process_video(input_source, youtube_url, file_path)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        return 0

    except KeyboardInterrupt:
        logger.info("Interrupted by user")
        return 1
    except Exception as e:
        logger.error(f"Fatal error: {e}", exc_info=True)
        return 1


if __name__ == "__main__":
    main()


YouTube SEO Optimization Engine



2026-02-20 13:33:16,117 [INFO] Starting YouTube SEO optimization
2026-02-20 13:33:16,372 [INFO] Downloading audio from YouTube: https://youtu.be/nIYFa2YF55U?si=Wtx7a5dwJxXWjfK3


2026-02-20 13:33:19,939 [INFO] Downloaded in 3.57s, duration: 139s, file: audio.m4a
2026-02-20 13:33:19,939 [INFO] Video duration: 139s (2m 19s)
2026-02-20 13:33:19,939 [INFO] Transcribing audio: C:\Temp\tmpnv5gnk49\audio.m4a
2026-02-20 13:33:37,086 [INFO] HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-02-20 13:33:37,086 [INFO] Transcription completed in 17.15s: 209 chars
2026-02-20 13:33:37,086 [INFO] Input processing completed in 20.96s
2026-02-20 13:33:39,875 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:33:39,886 [INFO] Agent 1 completed in 2.80s
2026-02-20 13:33:46,354 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:33:46,370 [INFO] Agent 2 completed in 6.48s
2026-02-20 13:33:57,487 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-20 13:33:57,487 [INFO] Agent 3 completed in 11.12s
2026-02

{
  "video_metadata": {
    "main_topic": "پاکستانی موسیقی کی تاریخ اور ثقافتی اثرات",
    "target_audience": "موسیقی کے شوقین، پاکستانی ثقافت میں دلچسپی رکھنے والے، طلباء اور محققین جو موسیقی کی تاریخ اور اس کے اثرات کے بارے میں معلومات تلاش کر رہے ہیں",
    "search_intent": "informational",
    "content_category": "Music",
    "duration_seconds": 139
  },
  "keywords": {
    "primary": "Pakistani music history and cultural impact",
    "secondary": [
      "best Pakistani music of all time",
      "top 10 Pakistani musicians you should know",
      "complete guide to Pakistani music genres",
      "how to appreciate Pakistani music",
      "ultimate guide to the evolution of Pakistani music",
      "best cultural influences in Pakistani music",
      "top historical events in Pakistani music",
      "Pakistani music trends in 2023",
      "must-visit places for Pakistani music lovers"
    ],
    "long_tail": [
      "how to understand the history of Pakistani music",
      "best Paki